# Tamil Handwritten Word Recognition (CRNN + CTC) on Kaggle GPU

**Project:** Tamil Handwritten Character Recognition and Unicode Conversion System  
**Branch:** `experiment/u14-iiit-indic-hw-uc`  
**Supported Datasets:**
1. **IIIT-Indic-HW-UC (Unconstrained) Tamil Dataset:** 206,635 words total (144,517 Train / 25,483 Val in 85/15 Document-Grouped Split, 30,000 Test Quarantined)
2. **IIIT-INDIC-HW-WORDS Tamil Dataset (Legacy):** 103,518 words (75,736 Train / 11,598 Val / 16,184 Test Quarantined)
**Architecture:** VGG-style 4-block CNN + 2-layer Bidirectional LSTM + CTC Head (73 classes: 72 Tamil chars + 1 blank)

---
### Pipeline Steps:
1. GPU & CUDA hardware acceleration verification.
2. Auto-discovery of dataset in `/kaggle/input/` (IIIT-Indic-HW-UC or Legacy Word Dataset) and setup in fast scratchpad.
3. Clone/pull the latest project repository (`main` branch).
4. Run unit tests (`test_dataset_hw_uc.py`, `test_crnn_dataset.py`, `test_full_page_ocr.py`).
5. Fast smoke test (2 batches) to verify AMP mixed precision and dataloader on GPU.
6. Full training loop with Resumable Checkpointing (`latest_checkpoint.pt`, `best_model.pt`).
7. Validation evaluation computing Character Error Rate (CER), Word Error Rate (WER), and Exact Match.
8. Bundle all artifacts into `tamil_crnn_artifacts.zip` with SHA-256 checksum for download.

In [ ]:
# Cell 1: Hardware & GPU Environment Verification
import os
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({total_mem:.2f} GB VRAM)")
    print(f"Device Count: {torch.cuda.device_count()}")
    !nvidia-smi
else:
    print("[WARNING] GPU acceleration is NOT detected! Please navigate to Notebook Settings -> Accelerator -> GPU T4 x2 or P100.")

In [ ]:
# Cell 2: Locate Dataset and Extract to Fast Scratchpad
import os
import glob
import zipfile
import shutil
from pathlib import Path

print("=== Searching for Tamil Datasets in /kaggle/input ===")

# Search for IIIT-Indic-HW-UC first
hw_uc_candidates = glob.glob('/kaggle/input/**/tamil_hw_uc*.zip', recursive=True) + \
                   glob.glob('/kaggle/input/**/iiit_indic_hw_uc*.zip', recursive=True)

# Check uncompressed IIIT-Indic-HW-UC candidates
hw_uc_dirs = glob.glob('/kaggle/input/**/train_gt.txt', recursive=True) + \
             glob.glob('/kaggle/input/**/train_doc_split_85_15.txt', recursive=True)

# Legacy candidates
legacy_zips = glob.glob('/kaggle/input/**/tamil_word_dataset.zip', recursive=True)
legacy_dirs = glob.glob('/kaggle/input/**/vocab.txt', recursive=True)

data_dir = '/kaggle/working/data'
os.makedirs(data_dir, exist_ok=True)
dataset_type = 'legacy'
train_gt_path = None
val_gt_path = None

if hw_uc_dirs:
    # Dataset already uncompressed on Kaggle
    file_dir = os.path.dirname(hw_uc_dirs[0])
    tamil_root = os.path.dirname(file_dir) if os.path.basename(file_dir) == 'file' else file_dir
    print(f"Found uncompressed IIIT-Indic-HW-UC at: {tamil_root}")
    data_dir = tamil_root
    dataset_type = 'hw_uc'
elif hw_uc_candidates:
    target_zip = hw_uc_candidates[0]
    print(f"Found IIIT-Indic-HW-UC archive: {target_zip} ({os.path.getsize(target_zip)/(1024**3):.2f} GB)")
    print(f"Unpacking to {data_dir}...")
    with zipfile.ZipFile(target_zip, 'r') as z:
        z.extractall(data_dir)
    # Locate tamil folder
    tamil_sub = os.path.join(data_dir, 'tamil')
    data_dir = tamil_sub if os.path.exists(tamil_sub) else data_dir
    dataset_type = 'hw_uc'
elif legacy_zips:
    target_zip = legacy_zips[0]
    print(f"Found legacy word dataset archive: {target_zip} ({os.path.getsize(target_zip)/(1024**2):.1f} MB)")
    with zipfile.ZipFile(target_zip, 'r') as root_z:
        for txt in ['vocab.txt', 'train.txt', 'val.txt']:
            if txt in root_z.namelist():
                root_z.extract(txt, data_dir)
        for split in ['val', 'train']:
            split_zip_name = f"{split}.zip"
            if split_zip_name in root_z.namelist():
                import io
                split_bytes = root_z.read(split_zip_name)
                with zipfile.ZipFile(io.BytesIO(split_bytes), 'r') as sz:
                    sz.extractall(data_dir)
    dataset_type = 'legacy'
elif legacy_dirs:
    data_dir = os.path.dirname(legacy_dirs[0])
    dataset_type = 'legacy'
else:
    raise FileNotFoundError("Could not locate IIIT-Indic-HW-UC or tamil_word_dataset in /kaggle/input!")

print(f"Active dataset type: {dataset_type}")
print(f"Active dataset path: {data_dir}")
!df -h /kaggle/working

In [ ]:
# Cell 3: Clone / Setup Git Repository (Experiment Branch)
%cd /kaggle/working
repo_dir = '/kaggle/working/tamil-handwritten-unicode'
branch_name = 'experiment/u14-iiit-indic-hw-uc'

if not os.path.exists(repo_dir):
    print(f"Cloning repository from branch: {branch_name}...")
    !git clone -b {branch_name} https://github.com/kamalakannan675/tamil-handwritten-unicode.git
else:
    print(f"Repository exists, pulling latest updates on {branch_name}...")
    %cd {repo_dir}
    !git fetch origin
    !git checkout {branch_name}
    !git pull origin {branch_name}

%cd {repo_dir}
print("=== Git Environment Verification ===")
!git status
!git rev-parse --short HEAD

# Preflight assertions
assert os.path.exists('src/data/dataset_hw_uc.py'), "Critical: src/data/dataset_hw_uc.py not found!"
assert os.path.exists('src/training/train_crnn.py'), "Critical: src/training/train_crnn.py not found!"

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

print("Repository configured and preflight checks passed successfully.")


In [ ]:
# Cell 4: Unit Test & Verification Suite
%cd {repo_dir}
print("=== Running CRNN & Dataset Unit Tests ===")
if os.path.exists('tests/test_dataset_hw_uc.py'):
    !python -m unittest tests.test_dataset_hw_uc
!python -m unittest tests.test_crnn_dataset
print("\n=== Running Full-Page OCR Tests ===")
!python -m unittest tests.test_full_page_ocr

In [ ]:
# Cell 5: Fast Smoke Test (2 Batches on GPU)
%cd {repo_dir}
print("=== Running Fast Smoke Test ===")

smoke_ckpt_dir = '/kaggle/working/outputs/checkpoints_smoke'
os.makedirs(smoke_ckpt_dir, exist_ok=True)

if dataset_type == 'hw_uc':
    # Check for document split file, otherwise fallback to train_gt.txt
    t_gt = os.path.join(data_dir, 'file', 'train_doc_split_85_15.txt')
    v_gt = os.path.join(data_dir, 'file', 'val_doc_split_85_15.txt')
    if not os.path.exists(t_gt):
        t_gt = os.path.join(data_dir, 'file', 'train_gt.txt')
        v_gt = os.path.join(data_dir, 'file', 'val_gt.txt')
    
    !python src/training/train_crnn.py \
        --dataset_type hw_uc \
        --hw_uc_root {data_dir} \
        --train_gt {t_gt} \
        --val_gt {v_gt} \
        --vocab_path checkpoints/crnn/tamil_ctc_char_vocab.json \
        --checkpoint_dir {smoke_ckpt_dir} \
        --batch_size 16 \
        --num_epochs 1 \
        --max_train_samples 32 \
        --max_val_samples 16
else:
    !python src/training/train_crnn.py \
        --extracted_dir {data_dir} \
        --vocab_path checkpoints/crnn/tamil_ctc_char_vocab.json \
        --checkpoint_dir {smoke_ckpt_dir} \
        --batch_size 16 \
        --num_epochs 1 \
        --max_train_samples 32 \
        --max_val_samples 16

print("Smoke test completed successfully.")

In [ ]:
# Cell 6: Full Model Training Loop
# Re-running this cell automatically resumes from latest_checkpoint.pt
%cd {repo_dir}
ckpt_dir = '/kaggle/working/outputs/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

if dataset_type == 'hw_uc':
    t_gt = os.path.join(data_dir, 'file', 'train_doc_split_85_15.txt')
    v_gt = os.path.join(data_dir, 'file', 'val_doc_split_85_15.txt')
    if not os.path.exists(t_gt):
        t_gt = os.path.join(data_dir, 'file', 'train_gt.txt')
        v_gt = os.path.join(data_dir, 'file', 'val_gt.txt')
    
    !python src/training/train_crnn.py \
        --dataset_type hw_uc \
        --hw_uc_root {data_dir} \
        --train_gt {t_gt} \
        --val_gt {v_gt} \
        --vocab_path checkpoints/crnn/tamil_ctc_char_vocab.json \
        --checkpoint_dir {ckpt_dir} \
        --batch_size 64 \
        --num_epochs 30 \
        --learning_rate 5e-4 \
        --hidden_size 256 \
        --patience 5 \
        --num_workers 2 \
        --resume
else:
    !python src/training/train_crnn.py \
        --extracted_dir {data_dir} \
        --vocab_path checkpoints/crnn/tamil_ctc_char_vocab.json \
        --checkpoint_dir {ckpt_dir} \
        --batch_size 64 \
        --num_epochs 30 \
        --learning_rate 5e-4 \
        --hidden_size 256 \
        --patience 5 \
        --num_workers 2 \
        --resume

In [ ]:
# Cell 7: Full Validation Partition Evaluation
# Note: Test sets remain quarantined to protect test integrity
%cd {repo_dir}
best_model_pt = f"{ckpt_dir}/best_model.pt"

print("=== Evaluating Best Model on Validation Partition ===")
if dataset_type == 'hw_uc':
    v_gt = os.path.join(data_dir, 'file', 'val_doc_split_85_15.txt')
    if not os.path.exists(v_gt):
        v_gt = os.path.join(data_dir, 'file', 'val_gt.txt')
    !python src/inference/evaluate_crnn.py \
        --checkpoint {best_model_pt} \
        --split val \
        --extracted_dir {data_dir} \
        --vocab_path checkpoints/crnn/tamil_ctc_char_vocab.json \
        --batch_size 64 \
        --output_json /kaggle/working/outputs/evaluation_crnn_val.json
else:
    !python src/inference/evaluate_crnn.py \
        --checkpoint {best_model_pt} \
        --split val \
        --extracted_dir {data_dir} \
        --vocab_path checkpoints/crnn/tamil_ctc_char_vocab.json \
        --batch_size 64 \
        --output_json /kaggle/working/outputs/evaluation_crnn_val.json

In [ ]:
# Cell 8: Bundle and Package Artifacts for Download
import zipfile
import hashlib

artifacts_zip = '/kaggle/working/tamil_crnn_artifacts.zip'
files_to_bundle = [
    f"{ckpt_dir}/best_model.pt",
    f"{ckpt_dir}/latest_checkpoint.pt",
    f"{ckpt_dir}/training_log.json",
    '/kaggle/working/outputs/evaluation_crnn_val.json',
    f"{repo_dir}/checkpoints/crnn/tamil_ctc_char_vocab.json"
]

print(f"Bundling training artifacts into {artifacts_zip}...")
with zipfile.ZipFile(artifacts_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in files_to_bundle:
        if os.path.exists(fpath):
            arcname = os.path.basename(fpath)
            zf.write(fpath, arcname=arcname)
            print(f"  Added: {arcname} ({os.path.getsize(fpath)/(1024**2):.2f} MB)")
        else:
            print(f"  [Notice] File not found (skipped): {fpath}")

# Calculate SHA-256 Checksum
sha256 = hashlib.sha256()
with open(artifacts_zip, 'rb') as f:
    for chunk in iter(lambda: f.read(65536), b''):
        sha256.update(chunk)
checksum = sha256.hexdigest()

print("\n======================================================")
print(f"Artifact Bundle Ready: {artifacts_zip}")
print(f"Size: {os.path.getsize(artifacts_zip)/(1024**2):.2f} MB")
print(f"SHA-256 Checksum: {checksum}")
print("Download this file from the Kaggle 'Output' tab on the right sidebar.")
print("======================================================")